# pyaegean — a guided tour

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryanpavlicek/pyaegean/blob/main/notebooks/getting-started.ipynb)

A short, runnable tour of **pyaegean**, a specialist Python toolkit for Ancient
Greek and the Aegean scripts. We answer two real questions — one in **Linear A**,
one in **Greek** — and let the library's objects render themselves as tables and
cards.

New to Python? See the [Getting Started guide](https://github.com/ryanpavlicek/pyaegean/wiki/Getting-Started). Run this notebook in your browser with the **Open in Colab** badge above — nothing to install.

In [ ]:
# pyaegean works fully offline once installed. This cell installs it if it
# isn't already available (e.g. the first time you open this in Google Colab).
try:
    import aegean
except ImportError:
    %pip install -q pyaegean
    import aegean

print("pyaegean", aegean.__version__, aegean.registered_scripts())

## 1. Linear A — does a 3,500-year-old ledger add up?

Many Linear A tablets are accounts: entries followed by a total word, **KU-RO**.
We load the bundled corpus and look at one tablet. (Objects display as rich
tables in Jupyter; outside a notebook they're ordinary Python values.)

In [2]:
corpus = aegean.load("lineara")   # 1,721 inscriptions, bundled, offline
corpus

id,site,period,words
HT1,Haghia Triada,LMIB,6
HT2,Haghia Triada,LMIB,2
HT3,Haghia Triada,LMIB,6
HT4,Haghia Triada,LMIB,4
HT5,Haghia Triada,LMIB,2
HT6a,Haghia Triada,LMIB,8
HT6b,Haghia Triada,LMIB,7
HT7a,Haghia Triada,LMIB,6


In [3]:
doc = corpus.get("HT13")          # a well-known account from Haghia Triada
doc

line,tokens
1,KA-U-DE-TA VIN 𐄁 TE 𐄁
2,RE-ZA 5 ¹⁄₂
3,TE-TU 56
4,TE-KI 27 ¹⁄₂
5,KU-ZU-NI 18
6,DA-SI-*118 19
7,I-DU-NE-SI 5
8,KU-RO 130 ¹⁄₂


`balance_check` sums the items a total governs and compares them to the stated
total. **This is exploratory** — section boundaries are heuristic and Linear A
metrology is contested; it flags lines worth a human's attention, not verdicts.

In [4]:
from aegean.analysis import balance_check

for chk in balance_check(doc):
    print(chk)

BalanceCheck(stated_total=130.5, computed_sum=131.0, item_count=6, difference=0.5, balances=False, marker='KU-RO', total_line_index=7)


Search for words shaped like `KU-?-RO`. The `*` wildcard means **exactly one**
sign in between (so `KU-RO` itself does not match).

In [5]:
from aegean.analysis import word_matches_sign_pattern

[(w, c) for w, c in corpus.word_frequencies()
 if word_matches_sign_pattern(w, "KU-*-RO")]

[('KU-MA-RO', 1)]

The query engine combines conditions — here, HT tablets that contain the word KU-RO:

In [6]:
from aegean.analysis import FilterRow, run_query

res = run_query(corpus, [
    FilterRow("id-contains", "HT"),
    FilterRow("ins-contains-word", "KU-RO", connector="and"),
], output="inscriptions")

len(res.inscriptions), [d.id for d in res.inscriptions][:8]

(32, ['HT9a', 'HT9b', 'HT11a', 'HT11b', 'HT13', 'HT25b', 'HT27a', 'HT39'])

The script's sign inventory renders as a table too:

In [7]:
aegean.get_script("lineara").sign_inventory

label,glyph,codepoint,phonetic
A,𐘇,67079,a
I,𐘚,67098,i
KA,𐘾,67134,ka
JA,𐘱,67121,ja
SI,𐘤,67108,si
TA,𐘳,67123,ta
DA,𐘀,67072,da
NA,𐘅,67077,na
SA,𐘞,67102,sa
MA,𐙁,67137,ma


## 2. Greek — reading the first line of the Odyssey

Now the Greek pipeline. Don't have a Greek keyboard? Type **Beta Code** and
convert with `greek.betacode_to_unicode(...)`. Here we scan Homer's opening line
into its metrical feet.

In [8]:
from aegean import greek

greek.scan_hexameter("ἄνδρα μοι ἔννεπε, Μοῦσα, πολύτροπον, ὃς μάλα πολλὰ")

foot,metre,syllables
dactyl,—⏑⏑,ἄν δρα μοι
dactyl,—⏑⏑,ἔν νε πε
dactyl,—⏑⏑,Μοῦ σα πο
dactyl,—⏑⏑,λύ τρο πον
dactyl,—⏑⏑,ὃς μά λα
final,—×,πολ λὰ


`analyze` returns the morphological readings an ending implies. On a **regular**
form it's strong; several readings come back when the ending is genuinely
ambiguous (you disambiguate with context).

In [9]:
greek.analyze("λόγον")[0]

Analysis(lemma='λόγος', pos='NOUN', case='acc', number='sg', gender='masc', tense=None, voice=None, mood=None, person=None, degree=None, lemma_certain=True)

In [10]:
for a in greek.analyze("λόγον"):
    print(a)

λόγος [NOUN acc sg masc]
λόγος [NOUN acc sg fem]
λόγος [NOUN nom sg neut]
λόγος [NOUN acc sg neut]
λόγος [NOUN voc sg neut]


## Where next

- [Tutorial](https://github.com/ryanpavlicek/pyaegean/wiki/Tutorial) — these two
  walkthroughs in prose, with more detail.
- [Greek NLP](https://github.com/ryanpavlicek/pyaegean/wiki/Greek-NLP) and
  [Analysis](https://github.com/ryanpavlicek/pyaegean/wiki/Analysis) — the full
  reference.
- [FAQ](https://github.com/ryanpavlicek/pyaegean/wiki/FAQ) — if something didn't
  go to plan.

Analysis of the undeciphered Linear A material is always **exploratory** — leads
for a human expert, never ground truth.